# Oxford-IIIT Pet: Fine-Grained Breed Classification
### 37 breeds | ResNet50 | Discriminative learning rates

We have ~7,349 cat and dog photos across 37 breeds — only about 150 images per breed,
less than Projects 1 or 3, spread across far more classes. And unlike "cat vs. dog vs.
snake," different breeds of the same species share almost everything (four legs, fur,
similar poses) and differ only in small, localized details: ear shape, muzzle length,
coat pattern. This is called **fine-grained classification**, and it's harder along
every axis at once — more classes, subtler cues, less data per class — which is why
this project reaches for a bigger backbone and a more careful fine-tuning strategy.

**Plan:**
1. Load the data and look at how little data we actually have per breed.
2. Use the same strong augmentation as Project 3, since data per class is even smaller.
3. Use **ResNet50** instead of ResNet18 — fine-grained detail needs more capacity.
4. Fine-tune with **discriminative learning rates**: every layer trains from epoch 1,
   but earlier layers move a tiny bit and later layers move more.
5. Evaluate by finding the *specific* breed pairs the model confuses, not just one
   overall accuracy number.

**Dataset:** `timm/oxford-iiit-pet` on Hugging Face — 7,349 images, 37 breeds. Run this
on Google Colab (GPU runtime recommended).

## 1. Setup

In [ ]:
!pip install -q datasets scikit-learn

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


## 2. Load and explore the dataset

Worth printing the 37 breed names and noting which ones sound like they might look
alike (two terrier breeds, several short-haired tabby-pattern cats) — a hypothesis to
check against the confusion matrix later.

In [ ]:
from datasets import load_dataset

raw = load_dataset("timm/oxford-iiit-pet")
print(raw)

# Carve train/val/test the same stratified way as earlier projects, using whichever
# splits the dataset actually ships.
if "validation" in raw or "val" in raw:
    train_full = raw["train"]
    test_ds = raw.get("validation", raw.get("val"))
else:
    train_full = raw["train"]
    test_ds = raw["test"] if "test" in raw else None

label_col = "label"
label_names = train_full.features[label_col].names
NUM_CLASSES = len(label_names)
print("num breeds:", NUM_CLASSES)
print(label_names)


In [ ]:
split = train_full.train_test_split(test_size=0.15, seed=SEED, stratify_by_column=label_col)
train_ds, val_ds = split["train"], split["test"]
if test_ds is None:
    split_2 = val_ds.train_test_split(test_size=0.5, seed=SEED, stratify_by_column=label_col)
    val_ds, test_ds = split_2["train"], split_2["test"]

print("train/val/test sizes:", len(train_ds), len(val_ds), len(test_ds))

counts = np.bincount([train_ds[i][label_col] for i in range(len(train_ds))], minlength=NUM_CLASSES)
print("min/median/max images per breed in train:", counts.min(), int(np.median(counts)), counts.max())
# Notice how few images per breed this is compared to Projects 1 and 3 -- fine-grained
# problems are hard partly because there is less data per class to learn subtler cues from.


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(15, 11))
idxs = random.sample(range(len(train_ds)), 12)
for ax, i in zip(axes.flat, idxs):
    ex = train_ds[i]
    ax.imshow(ex["image"]); ax.set_title(label_names[ex[label_col]], fontsize=9); ax.axis("off")
plt.suptitle("Random training samples across 37 breeds")
plt.tight_layout(); plt.show()


## 3. Preprocessing and augmentation

Same augmentation philosophy as Project 3 (`RandomResizedCrop`, color jitter,
`RandomErasing`) — with even less data per class here, augmentation matters at least as
much. Same ImageNet mean/std normalization the pretrained backbone expects.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.08)),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


class HFPetDataset(Dataset):
    def __init__(self, hf_dataset, transform, label_col="label"):
        self.ds, self.transform, self.label_col = hf_dataset, transform, label_col

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        ex = self.ds[idx]
        return self.transform(ex["image"].convert("RGB")), ex[self.label_col]


BATCH_SIZE = 32
train_loader = DataLoader(HFPetDataset(train_ds, train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(HFPetDataset(val_ds, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(HFPetDataset(test_ds, eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"train/val/test batches: {len(train_loader)}/{len(val_loader)}/{len(test_loader)}")


## 4. Why ResNet50 over VGG16 for this task

| Model | Params | ImageNet top-1 |
|---|---|---|
| **ResNet50** | 25.6M | ~76.1% (chosen) |
| VGG16 | 138M | ~71.6% |

This is the mirror image of Project 1's ResNet18-vs-VGG16 choice — the *reasoning*
("match capacity to task difficulty") is the same, it just points to a bigger model
this time because the task is harder:

1. Fine-grained discrimination needs a deeper, more expressive feature hierarchy than a
   coarse 3-class split does — ResNet50's extra depth over ResNet18 earns its keep here.
2. We're about to fine-tune much more of the network than earlier projects.
   ResNet50's residual connections keep gradients well-behaved when a large fraction of
   the network updates at once; VGG16's plain stacking (no skip connections) doesn't
   handle that as reliably.
3. Most of VGG16's 138M parameters sit in a huge 4096-4096 fully-connected head — prone
   to overfitting on ~150 images/class. ResNet50 replaces that with one small linear
   layer after global average pooling.

## 5. Build the model, and set up discriminative learning rates

Projects 1 and 3 both used a clean two-phase recipe: freeze everything, train the head,
then unfreeze exactly one block at a lower learning rate. For a genuinely fine-grained
task we want the *whole* backbone to adapt a little, not an all-or-nothing cutoff at one
block boundary. PyTorch lets one optimizer apply a whole list of `{params, lr}` groups
at once — every layer trains from epoch 1, at a rate matched to how much we trust its
current weights:

```text
conv1 + layer1  (earliest, most generic)     lr = 1e-6   barely move
layer2                                        lr = 1e-5
layer3                                        lr = 3e-5
layer4          (closest to output, specific) lr = 1e-4
fc              (brand new, random init)      lr = 1e-3   move freely
```

The rule is the same one from Project 1's freeze/unfreeze choice, just applied more
finely: the more valuable the pretrained weights already are, the smaller the learning
rate they get. `fc` has no pretrained knowledge to protect, so it gets the largest rate.

In [ ]:
def build_resnet50(num_classes):
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model  # nothing frozen -- every parameter trains, but at DIFFERENT rates.


model = build_resnet50(NUM_CLASSES).to(DEVICE)

param_groups = [
    {"params": model.conv1.parameters(), "lr": 1e-6},
    {"params": model.bn1.parameters(), "lr": 1e-6},
    {"params": model.layer1.parameters(), "lr": 1e-6},
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 3e-5},
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(), "lr": 1e-3},
]

for g in param_groups:
    n_params = sum(p.numel() for p in g["params"])
    print(f"lr={g['lr']:.0e}  params={n_params:,}")

optimizer = optim.Adam(param_groups)
criterion = nn.CrossEntropyLoss()
# Adam still adapts each parameter's step size internally on top of this -- the group lr
# sets a ceiling per layer, Adam fine-tunes further within it. The two are complementary.


## 6. Train

Discriminative LRs let us safely train the whole network at once from epoch 1, instead
of needing separate frozen and fine-tuning phases — most of the "phase 1, then phase 2"
structure from earlier projects is now baked directly into the per-layer rates, which is
why the total epoch budget (8) lands close to what Projects 1/3 used across *both*
phases combined, not something that scales up with the number of classes.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    torch.set_grad_enabled(is_train)
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if is_train:
            optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        if is_train:
            loss.backward(); optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        n += images.size(0)
    torch.set_grad_enabled(True)
    return total_loss / n, correct / n


EPOCHS = 8
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
for epoch in range(1, EPOCHS + 1):
    tl, ta = run_epoch(model, train_loader, criterion, optimizer)
    vl, va = run_epoch(model, val_loader, criterion, None)
    history["train_loss"].append(tl); history["train_acc"].append(ta)
    history["val_loss"].append(vl); history["val_acc"].append(va)
    print(f"epoch {epoch}/{EPOCHS} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {va:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs_range = range(1, EPOCHS + 1)
axes[0].plot(epochs_range, history["train_loss"], marker="o", label="train")
axes[0].plot(epochs_range, history["val_loss"], marker="o", label="val")
axes[0].set_title("Loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].plot(epochs_range, history["train_acc"], marker="o", label="train")
axes[1].plot(epochs_range, history["val_acc"], marker="o", label="val")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout(); plt.show()
# With 37 classes and ~150 images/class, some train/val gap is expected. The concern
# would be a gap that keeps WIDENING every epoch -- that means stop earlier or add
# more regularization/augmentation.


## 7. Evaluation: overall metrics + which specific breeds get confused

A single accuracy number hides a lot with 37 classes — some breed pairs are
near-perfectly separable, others are genuinely hard even for a human. A 37×37 confusion
matrix would also be too dense to read as a heatmap the way Project 3's 7×7 one was, so
instead we extract and rank the specific pairs the model confuses most.

In [ ]:
@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    for images, labels in loader:
        preds = model(images.to(DEVICE)).argmax(1).cpu()
        all_preds.append(preds); all_labels.append(labels)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()


test_preds, test_labels = collect_predictions(model, test_loader)
test_acc = (test_preds == test_labels).mean()
print(f"TEST accuracy (37 breeds): {test_acc:.4f}\n")
print(classification_report(test_labels, test_preds, target_names=label_names, digits=3))


In [ ]:
# Instead of a dense, unreadable 37x37 heatmap, extract the top-15 most-confused
# (true, predicted) breed pairs directly from the confusion matrix.
cm = confusion_matrix(test_labels, test_preds)
np.fill_diagonal(cm, 0)  # zero out correct predictions -- we only want to rank ERRORS

pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] > 0:
            pairs.append((cm[i, j], label_names[i], label_names[j]))
pairs.sort(reverse=True)

print(f"{'count':>6s}  {'true breed':30s}  {'predicted as':30s}")
for count, true_name, pred_name in pairs[:15]:
    print(f"{count:6d}  {true_name:30s}  {pred_name:30s}")
# The top pairs are likely to be visually similar breeds. Cross-check a few against the
# sample images from Section 2 -- that separates "a genuinely hard, defensible mistake"
# from "something is off with a specific breed's data".


In [ ]:
# Visually compare a handful of images from the single most-confused pair.
if pairs:
    _, true_name, pred_name = pairs[0]
    true_idx = label_names.index(true_name)
    pred_idx = label_names.index(pred_name)

    true_examples = [i for i in range(len(test_ds)) if test_ds[i][label_col] == true_idx][:4]
    pred_examples = [i for i in range(len(test_ds)) if test_ds[i][label_col] == pred_idx][:4]

    fig, axes = plt.subplots(2, 4, figsize=(14, 7))
    for j, (ax, i) in enumerate(zip(axes[0], true_examples)):
        ax.imshow(test_ds[i]["image"]); ax.axis("off")
        if j == 0:
            ax.set_title(true_name, fontsize=10, loc="left")
    for j, (ax, i) in enumerate(zip(axes[1], pred_examples)):
        ax.imshow(test_ds[i]["image"]); ax.axis("off")
        if j == 0:
            ax.set_title(pred_name, fontsize=10, loc="left")
    plt.suptitle(f"Most-confused pair: '{true_name}' (top row) vs '{pred_name}' (bottom row)")
    plt.tight_layout(); plt.show()


## 8. Ideas to improve

- Targeted data/augmentation for the top confused pairs — the highest-leverage move now
  that we know exactly which breeds are hard.
- Higher input resolution (320-380px) — breed-distinguishing details (ear shape, coat
  pattern) are small and localized, so more pixels helps more here than in Projects 1/3.
- Tune the discriminative-LR schedule itself — try `CosineAnnealingLR`/`OneCycleLR` on
  top of the per-group base rates, or adjust the ratios between layer groups.
- Segment out the pet from the background first (the dataset ships trimap masks) —
  removes a source of noise unrelated to the breed-distinguishing features.